In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/pill_detection'
DATA_DIR = f'{BASE}/data'
TRAIN_IMG_DIR = f'{DATA_DIR}/train_images'
TRAIN_ANN_DIR = f'{DATA_DIR}/train_annotations'
TEST_IMG_DIR = f'{DATA_DIR}/test_images'
OUTPUT_DIR = f'{BASE}/outputs'
YOLO_DIR = f'{BASE}/yolo_data'
yaml_path = f'{YOLO_DIR}/data.yaml'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

!pip install ultralytics ensemble-boxes -q

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.2 MB/s eta 0:00:00


In [ ]:
import glob, json, csv, os
import pandas as pd
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion

# category_id 매핑
json_files = glob.glob(f'{TRAIN_ANN_DIR}/**/*.json', recursive=True)
dlid_set = set()
for jf in json_files:
    with open(jf, 'r', encoding='utf-8') as f:
        data = json.load(f)
    for img_info in data['images']:
        dlid_set.add(int(img_info['dl_idx']))

dlid_to_idx = {dlid: i+1 for i, dlid in enumerate(sorted(dlid_set))}
idx_to_catid = {v: k for k, v in dlid_to_idx.items()}
print(f"매핑 완료: {len(idx_to_catid)}개")

# 예측 함수
def predict_with_tta(model_path, model_name, conf=0.25):
    model = YOLO(model_path)
    test_imgs = sorted([f for f in os.listdir(TEST_IMG_DIR) if f.endswith('.png')])
    rows = []
    ann_id = 1
    for img_file in test_imgs:
        image_id = int(img_file.replace('.png', ''))
        img_path = os.path.join(TEST_IMG_DIR, img_file)
        results = model.predict(img_path, conf=conf, iou=0.5, augment=True, verbose=False)
        for r in results:
            if r.boxes is None:
                continue
            for box in r.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                w, h = x2 - x1, y2 - y1
                score = float(box.conf[0])
                cat_id = idx_to_catid[int(box.cls[0]) + 1]
                rows.append([ann_id, image_id, cat_id,
                              round(x1,2), round(y1,2),
                              round(w,2), round(h,2),
                              round(score,4)])
                ann_id += 1
    out_path = f'{OUTPUT_DIR}/pred_{model_name}.csv'
    with open(out_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['annotation_id','image_id','category_id',
                         'bbox_x','bbox_y','bbox_w','bbox_h','score'])
        writer.writerows(rows)
    print(f"[{model_name}] {len(rows)} rows → {out_path}")
    return out_path

# WBF 앙상블 함수
def run_wbf_ensemble(pred_paths, weights, iou_thr=0.55, skip_box_thr=0.2, out_name='ensemble'):
    dfs = [pd.read_csv(p) for p in pred_paths]
    all_image_ids = sorted(set().union(*[set(df['image_id']) for df in dfs]))
    IMG_W, IMG_H = 976, 1280
    rows = []
    ann_id = 1
    for img_id in all_image_ids:
        boxes_list, scores_list, labels_list = [], [], []
        for df in dfs:
            sub = df[df['image_id'] == img_id]
            if len(sub) == 0:
                boxes_list.append([]); scores_list.append([]); labels_list.append([])
                continue
            bboxes = []
            for _, row in sub.iterrows():
                x1 = row['bbox_x'] / IMG_W
                y1 = row['bbox_y'] / IMG_H
                x2 = (row['bbox_x'] + row['bbox_w']) / IMG_W
                y2 = (row['bbox_y'] + row['bbox_h']) / IMG_H
                bboxes.append([max(0,min(1,x1)), max(0,min(1,y1)),
                                max(0,min(1,x2)), max(0,min(1,y2))])
            boxes_list.append(bboxes)
            scores_list.append(sub['score'].tolist())
            labels_list.append(sub['category_id'].tolist())
        boxes, scores, labels = weighted_boxes_fusion(
            boxes_list, scores_list, labels_list,
            weights=weights, iou_thr=iou_thr, skip_box_thr=skip_box_thr
        )
        for box, score, label in zip(boxes, scores, labels):
            x1 = box[0] * IMG_W
            y1 = box[1] * IMG_H
            w  = (box[2] - box[0]) * IMG_W
            h  = (box[3] - box[1]) * IMG_H
            rows.append([ann_id, img_id, int(label),
                         round(x1,2), round(y1,2),
                         round(w,2), round(h,2),
                         round(float(score),4)])
            ann_id += 1
    out_path = f'{OUTPUT_DIR}/{out_name}.csv'
    with open(out_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['annotation_id','image_id','category_id',
                         'bbox_x','bbox_y','bbox_w','bbox_h','score'])
        writer.writerows(rows)
    print(f"[{out_name}] {len(rows)} rows → {out_path}")
    return out_path

# 기존 pred CSV 경로
pred_8s  = f'{OUTPUT_DIR}/pred_v8s.csv'
pred_8m  = f'{OUTPUT_DIR}/pred_v8m.csv'
pred_11s = f'{OUTPUT_DIR}/pred_v11s.csv'
pred_11s_aug = f'{OUTPUT_DIR}/pred_v11s_aug.csv'
pred_11m = f'{OUTPUT_DIR}/pred_v11m.csv'

print("세팅 완료!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
매핑 완료: 56개
세팅 완료!


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11l.pt')

results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=1024,
    batch=4,
    project=f'{BASE}/runs',
    name='yolo11l_exp',
    optimizer='AdamW',
    lr0=0.001,
    warmup_epochs=3,
    cos_lr=True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=15,
    flipud=0.3,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    cls=1.5,
    patience=20,
    save=True,
    device=0
)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/pill_detection/yolo_data/data.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11l_exp, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, 

In [ ]:
pred_11l = predict_with_tta(
    f'{BASE}/runs/yolo11l_exp/weights/best.pt',
    'v11l'
)
print("v11l 예측 완료!")

[v11l] 3763 rows → /content/drive/MyDrive/pill_detection/outputs/pred_v11l.csv
v11l 예측 완료!


In [ ]:
pred_11l = f'{OUTPUT_DIR}/pred_v11l.csv'

# 오늘 실험 조합
today_experiments = [
    # (파일명, pred_paths, weights, iou_thr, skip_box_thr)
    ('v11l_solo',        [pred_11l],                              [1.0],              0.55, 0.20),
    ('v11l_4models',     [pred_8s, pred_8m, pred_11s, pred_11l], [1.0,1.0,1.0,1.0], 0.55, 0.20),
    ('v11l_4models_boost',[pred_8s, pred_8m, pred_11s, pred_11l],[0.8,1.0,1.0,1.2], 0.55, 0.20),
    ('v11l_5models',     [pred_8s, pred_8m, pred_11s, pred_11m, pred_11l],[1.0,1.0,1.0,1.0,1.0], 0.55, 0.20),
    ('v11l_3models_new', [pred_8m, pred_11s, pred_11l],          [1.0,1.0,1.0],      0.55, 0.20),
    ('v11m_v11l',        [pred_11m, pred_11l],                   [1.0,1.0],          0.55, 0.20),
    ('v11s_v11m_v11l',   [pred_11s, pred_11m, pred_11l],         [1.0,1.0,1.0],      0.55, 0.20),
    ('v11l_aug_5models', [pred_8s, pred_8m, pred_11s, pred_11s_aug, pred_11l],[1.0,1.0,1.0,1.0,1.0], 0.55, 0.20),
    ('best3_vs_v11l',    [pred_8s, pred_8m, pred_11s],           [1.0,1.0,1.0],      0.55, 0.20),
    ('v11l_heavy',       [pred_8s, pred_8m, pred_11s, pred_11l], [0.5,0.8,1.0,1.5], 0.55, 0.20),
]

for name, paths, weights, iou, skip in today_experiments:
    run_wbf_ensemble(paths, weights, iou_thr=iou, skip_box_thr=skip, out_name=f'today_{name}')

[today_v11l_solo] 3763 rows → /content/drive/MyDrive/pill_detection/outputs/today_v11l_solo.csv
[today_v11l_4models] 4550 rows → /content/drive/MyDrive/pill_detection/outputs/today_v11l_4models.csv
[today_v11l_4models_boost] 4550 rows → /content/drive/MyDrive/pill_detection/outputs/today_v11l_4models_boost.csv
[today_v11l_5models] 4754 rows → /content/drive/MyDrive/pill_detection/outputs/today_v11l_5models.csv
[today_v11l_3models_new] 4266 rows → /content/drive/MyDrive/pill_detection/outputs/today_v11l_3models_new.csv
[today_v11m_v11l] 4051 rows → /content/drive/MyDrive/pill_detection/outputs/today_v11m_v11l.csv
[today_v11s_v11m_v11l] 4243 rows → /content/drive/MyDrive/pill_detection/outputs/today_v11s_v11m_v11l.csv
[today_v11l_aug_5models] 4726 rows → /content/drive/MyDrive/pill_detection/outputs/today_v11l_aug_5models.csv
[today_best3_vs_v11l] 4160 rows → /content/drive/MyDrive/pill_detection/outputs/today_best3_vs_v11l.csv
[today_v11l_heavy] 4550 rows → /content/drive/MyDrive/pill_d

In [ ]:
import pandas as pd
df = pd.read_csv(f'{OUTPUT_DIR}/today_v11l_solo.csv')
print(df['category_id'].head(5).tolist())

[27926, 1900, 29345, 16551, 1900]


In [ ]:
# conf=0.15로 재예측
pred_8s_c15  = predict_with_tta(f'{BASE}/runs/baseline_yolov8s/weights/best.pt',  'v8s_c15',  conf=0.15)
pred_8m_c15  = predict_with_tta(f'{BASE}/runs/yolov8m_exp/weights/best.pt',       'v8m_c15',  conf=0.15)
pred_11s_c15 = predict_with_tta(f'{BASE}/runs/yolo11s_exp/weights/best.pt',       'v11s_c15', conf=0.15)

[v8s_c15] 3676 rows → /content/drive/MyDrive/pill_detection/outputs/pred_v8s_c15.csv
[v8m_c15] 3624 rows → /content/drive/MyDrive/pill_detection/outputs/pred_v8m_c15.csv
[v11s_c15] 3471 rows → /content/drive/MyDrive/pill_detection/outputs/pred_v11s_c15.csv


In [ ]:
pred_8s_c15  = f'{OUTPUT_DIR}/pred_v8s_c15.csv'
pred_8m_c15  = f'{OUTPUT_DIR}/pred_v8m_c15.csv'
pred_11s_c15 = f'{OUTPUT_DIR}/pred_v11s_c15.csv'

conf15_experiments = [
    ('c15_equal',      [pred_8s_c15, pred_8m_c15, pred_11s_c15], [1.0,1.0,1.0], 0.55, 0.20),
    ('c15_skip15',     [pred_8s_c15, pred_8m_c15, pred_11s_c15], [1.0,1.0,1.0], 0.55, 0.15),
    ('c15_iou50',      [pred_8s_c15, pred_8m_c15, pred_11s_c15], [1.0,1.0,1.0], 0.50, 0.20),
    ('c15_mix_best3',  [pred_8s, pred_8m, pred_11s,
                        pred_8s_c15, pred_8m_c15, pred_11s_c15], [1.0,1.0,1.0,1.0,1.0,1.0], 0.55, 0.20),
]

for name, paths, weights, iou, skip in conf15_experiments:
    run_wbf_ensemble(paths, weights, iou_thr=iou, skip_box_thr=skip, out_name=f'wbf_{name}')

[wbf_c15_equal] 4270 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_c15_equal.csv
[wbf_c15_skip15] 4422 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_c15_skip15.csv
[wbf_c15_iou50] 4270 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_c15_iou50.csv
[wbf_c15_mix_best3] 4270 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_c15_mix_best3.csv


In [ ]:
df = pd.read_csv(f'{OUTPUT_DIR}/wbf_c15_equal.csv')
print(df['category_id'].head(5).tolist())

[27926, 1900, 16551, 29345, 27733]


In [ ]:
from ultralytics import YOLO

# 1번 - YOLOv11s imgsz=1280 재학습 (약 1시간 30분)
model1 = YOLO('yolo11s.pt')
model1.train(
    data=yaml_path,
    epochs=150,
    imgsz=1280,
    batch=2,        # 1280이라 batch 줄여야 함
    project=f'{BASE}/runs',
    name='yolo11s_1280',
    optimizer='AdamW',
    lr0=0.001,
    warmup_epochs=3,
    cos_lr=True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=15,
    flipud=0.3,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.15,
    cls=1.5,
    patience=30,
    save=True,
    device=0
)

# 2번 - YOLOv8m imgsz=1280 재학습 (약 2시간)
model2 = YOLO('yolov8m.pt')
model2.train(
    data=yaml_path,
    epochs=150,
    imgsz=1280,
    batch=2,
    project=f'{BASE}/runs',
    name='yolov8m_1280',
    optimizer='AdamW',
    lr0=0.001,
    warmup_epochs=3,
    cos_lr=True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=15,
    flipud=0.3,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.15,
    cls=1.5,
    patience=30,
    save=True,
    device=0
)

# 3번 - YOLOv11s 증강 강화 재학습 (약 1시간 30분)
model3 = YOLO('yolo11s.pt')
model3.train(
    data=yaml_path,
    epochs=150,
    imgsz=1024,
    batch=4,
    project=f'{BASE}/runs',
    name='yolo11s_aug_v2',
    optimizer='AdamW',
    lr0=0.001,
    warmup_epochs=3,
    cos_lr=True,
    hsv_h=0.02, hsv_s=0.9, hsv_v=0.5,
    degrees=30,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.2,
    cls=1.5,
    patience=30,
    save=True,
    device=0
)

print("전체 학습 완료!")

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.15, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/pill_detection/yolo_data/data.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11s_1280-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=Ad

In [ ]:
pred_11s_1280 = predict_with_tta(
    f'{BASE}/runs/yolo11s_1280-2/weights/best.pt', 'v11s_1280')

pred_8m_1280 = predict_with_tta(
    f'{BASE}/runs/yolov8m_1280/weights/best.pt', 'v8m_1280')

pred_11s_augv2 = predict_with_tta(
    f'{BASE}/runs/yolo11s_aug_v2/weights/best.pt', 'v11s_augv2')

[v11s_1280] 3407 rows → /content/drive/MyDrive/pill_detection/outputs/pred_v11s_1280.csv
[v8m_1280] 3717 rows → /content/drive/MyDrive/pill_detection/outputs/pred_v8m_1280.csv
[v11s_augv2] 3386 rows → /content/drive/MyDrive/pill_detection/outputs/pred_v11s_augv2.csv


In [ ]:
pred_11s_1280  = f'{OUTPUT_DIR}/pred_v11s_1280.csv'
pred_8m_1280   = f'{OUTPUT_DIR}/pred_v8m_1280.csv'
pred_11s_augv2 = f'{OUTPUT_DIR}/pred_v11s_augv2.csv'

today_experiments = [
    # (파일명, pred_paths, weights, iou_thr, skip_box_thr)
    ('augv2_solo',       [pred_11s_augv2],                                          [1.0],                  0.55, 0.20),
    ('augv2_4models',    [pred_8s, pred_8m, pred_11s, pred_11s_augv2],              [1.0,1.0,1.0,1.0],     0.55, 0.20),
    ('augv2_5models',    [pred_8s, pred_8m, pred_11s, pred_11s_augv2, pred_11s_1280],[1.0,1.0,1.0,1.0,1.0],0.55, 0.20),
    ('augv2_2models',    [pred_11s, pred_11s_augv2],                                [1.0,1.0],              0.55, 0.20),
    ('augv2_replace',    [pred_8s, pred_8m, pred_11s_augv2],                        [1.0,1.0,1.0],          0.55, 0.20),
    ('1280_solo',        [pred_11s_1280],                                            [1.0],                  0.55, 0.20),
    ('augv2_4m_skip15',  [pred_8s, pred_8m, pred_11s, pred_11s_augv2],              [1.0,1.0,1.0,1.0],     0.55, 0.15),
]

for name, paths, weights, iou, skip in today_experiments:
    run_wbf_ensemble(paths, weights, iou_thr=iou, skip_box_thr=skip, out_name=f'wbf_{name}')

[wbf_augv2_solo] 3386 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_augv2_solo.csv
[wbf_augv2_4models] 4315 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_augv2_4models.csv
[wbf_augv2_5models] 4390 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_augv2_5models.csv
[wbf_augv2_2models] 3670 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_augv2_2models.csv
[wbf_augv2_replace] 4154 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_augv2_replace.csv
[wbf_1280_solo] 3407 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_1280_solo.csv
[wbf_augv2_4m_skip15] 4315 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_augv2_4m_skip15.csv


In [ ]:
pred_11s_1280 = f'{OUTPUT_DIR}/pred_v11s_1280.csv'
pred_8m_1280  = f'{OUTPUT_DIR}/pred_v8m_1280.csv'

new_experiments = [
    ('3m_plus_1280s',  [pred_8s, pred_8m, pred_11s, pred_11s_1280],            [1.0,1.0,1.0,1.0], 0.55, 0.20),
    ('3m_plus_1280m',  [pred_8s, pred_8m, pred_11s, pred_8m_1280],             [1.0,1.0,1.0,1.0], 0.55, 0.20),
    ('1280_combo',     [pred_11s, pred_11s_1280, pred_8m_1280],                 [1.0,1.0,1.0],     0.55, 0.20),
    ('best3_skip25',   [pred_8s, pred_8m, pred_11s],                            [1.0,1.0,1.0],     0.55, 0.25),
]

for name, paths, weights, iou, skip in new_experiments:
    run_wbf_ensemble(paths, weights, iou_thr=iou, skip_box_thr=skip, out_name=f'wbf_{name}')

[wbf_3m_plus_1280s] 4301 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_3m_plus_1280s.csv
[wbf_3m_plus_1280m] 4738 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_3m_plus_1280m.csv
[wbf_1280_combo] 4330 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_1280_combo.csv
[wbf_best3_skip25] 4160 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_best3_skip25.csv
